In [1]:
"""
MDB Financing — A Hands-On Math Primer
======================================
A small, dependency-free toolkit illustrating the financial mathematics
embedded in Multilateral Development Bank (MDB) instruments, plus a simple
IMF rate-of-charge calculator for contrast.

Pure standard library — just run:  python3 mdb_finance_primer.py

Concepts covered
----------------
1. Funding-to-lending spread (the MDB intermediation engine)
2. Grant element  (concessionality of a loan)  -- DAC-style + bullet loan
3. Bond pricing and yield-to-maturity (how MDBs raise money)
4. Blended-finance leverage / mobilization ratio
5. IMF rate of charge (SDR rate + margin + surcharges) -- the contrast

NOTE ON CONVENTIONS: real DAC grant-element calculations use semiannual
payments and differentiated discount rates by income group. Here the
discount rate and payment frequency are parameters so you can match
whatever convention you need.
"""

from __future__ import annotations


# ---------------------------------------------------------------------------
# 1. The intermediation engine: funding cost vs lending rate
# ---------------------------------------------------------------------------
def funding_spread(bond_yield: float, lending_rate: float) -> float:
    """Gross margin the MDB earns: it borrows at `bond_yield`, lends higher.

    All rates are annual decimals (0.04 == 4%). Returns the spread in bps.
    """
    return (lending_rate - bond_yield) * 10_000  # basis points


# ---------------------------------------------------------------------------
# 2. Grant element: how much of a "loan" is really a gift
# ---------------------------------------------------------------------------
def grant_element_amortizing(
    face: float,
    rate: float,
    grace_years: float,
    maturity_years: float,
    discount: float,
    freq: int = 2,
) -> float:
    """Grant element (%) for a loan with a grace period and equal principal
    repayments after grace. Interest is paid (not capitalized) during grace.

    face          : principal / face value
    rate          : annual interest rate on the loan (decimal)
    grace_years   : years of grace (principal deferred, interest still paid)
    maturity_years: total tenor in years
    discount      : reference discount rate (decimal) e.g. 0.05 or 0.10
    freq          : payments per year (DAC convention = 2)

    GE > 0 when rate < discount. Higher grace/maturity -> higher GE.
    """
    n_total = round(maturity_years * freq)
    n_grace = round(grace_years * freq)
    n_repay = n_total - n_grace
    per_rate = rate / freq
    per_disc = discount / freq
    principal_pmt = face / n_repay

    balance = face
    pv_debt_service = 0.0
    for t in range(1, n_total + 1):
        interest = balance * per_rate
        principal = principal_pmt if t > n_grace else 0.0
        debt_service = interest + principal
        pv_debt_service += debt_service / (1 + per_disc) ** t
        balance -= principal

    return (1 - pv_debt_service / face) * 100


def grant_element_bullet(
    face: float, rate: float, maturity_years: int, discount: float
) -> float:
    """Grant element (%) for a simple bullet loan: interest annually, full
    principal repaid at maturity. Matches the hand-worked chat example.
    """
    annuity_factor = (1 - (1 + discount) ** -maturity_years) / discount
    pv_interest = face * rate * annuity_factor
    pv_principal = face / (1 + discount) ** maturity_years
    pv_debt_service = pv_interest + pv_principal
    return (1 - pv_debt_service / face) * 100


# ---------------------------------------------------------------------------
# 3. Bond pricing & yield-to-maturity (the MDB funding side)
# ---------------------------------------------------------------------------
def bond_price(
    face: float, coupon_rate: float, years: int, ytm: float, freq: int = 2
) -> float:
    """Clean price of a coupon bond = PV of coupons + PV of face."""
    n = years * freq
    coupon = face * coupon_rate / freq
    per_ytm = ytm / freq
    pv_coupons = sum(coupon / (1 + per_ytm) ** t for t in range(1, n + 1))
    pv_face = face / (1 + per_ytm) ** n
    return pv_coupons + pv_face


def bond_ytm(
    price: float, face: float, coupon_rate: float, years: int, freq: int = 2
) -> float:
    """Solve for yield-to-maturity by bisection. Returns annual decimal."""
    lo, hi = 1e-6, 1.0  # 0% to 100%
    for _ in range(200):
        mid = (lo + hi) / 2
        if bond_price(face, coupon_rate, years, mid, freq) > price:
            lo = mid  # price too high -> need higher yield
        else:
            hi = mid
    return (lo + hi) / 2


# ---------------------------------------------------------------------------
# 4. Blended-finance leverage / mobilization ratio
# ---------------------------------------------------------------------------
def mobilization_ratio(public_input: float, total_capital: float) -> float:
    """How many dollars of total capital each public dollar mobilizes.
    private_mobilized = total_capital - public_input
    ratio = private_mobilized / public_input
    """
    private_mobilized = total_capital - public_input
    return private_mobilized / public_input


# ---------------------------------------------------------------------------
# 5. IMF rate of charge (the contrast: monetary, not development, finance)
# ---------------------------------------------------------------------------
def imf_rate_of_charge(
    sdr_rate: float,
    margin: float = 0.01,
    level_surcharge: float = 0.0,
    time_surcharge: float = 0.0,
) -> float:
    """Illustrative IMF basic rate of charge.

    The IMF prices off the SDR interest rate plus a margin (recently ~100 bps),
    with additional surcharges for large (level) or prolonged (time) borrowing.
    Unlike an MDB it is NOT funding this from bond issuance — it draws on the
    pooled quota resources of its members.
    """
    return sdr_rate + margin + level_surcharge + time_surcharge


# ---------------------------------------------------------------------------
# Worked examples
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    line = "-" * 68

    print(line)
    print("1. FUNDING-TO-LENDING SPREAD (the intermediation engine)")
    print(line)
    by, lr = 0.035, 0.052  # MDB issues bonds at 3.5%, lends at 5.2%
    print(f"  Bond yield (cost of funds): {by:.2%}")
    print(f"  Lending rate to borrower:   {lr:.2%}")
    print(f"  Gross spread:               {funding_spread(by, lr):.0f} bps")

    print()
    print(line)
    print("2. GRANT ELEMENT (concessionality)")
    print(line)
    # Bullet loan matching the hand-worked chat example
    ge_bullet = grant_element_bullet(100, 0.02, 10, 0.06)
    print("  Bullet loan: $100M, 2% rate, 10y, repaid at maturity, disc 6%")
    print(f"    Grant element: {ge_bullet:.1f}%   (~29 cents/$ is a gift)")
    print()
    # IDA-style highly concessional loan
    ge_ida = grant_element_amortizing(
        face=100, rate=0.01, grace_years=10, maturity_years=40,
        discount=0.05, freq=2,
    )
    print("  IDA-style: $100M, 1% rate, 10y grace, 40y maturity, disc 5%")
    print(f"    Grant element: {ge_ida:.1f}%")
    print()
    # Near-market loan: little to no concessionality
    ge_market = grant_element_amortizing(
        face=100, rate=0.05, grace_years=0, maturity_years=15,
        discount=0.05, freq=2,
    )
    print("  Near-market: $100M, 5% rate, no grace, 15y, disc 5%")
    print(f"    Grant element: {ge_market:.1f}%   (~0 -> not concessional)")

    print()
    print(line)
    print("3. BOND PRICING & YIELD (how the MDB raises money)")
    print(line)
    price = bond_price(face=1000, coupon_rate=0.03, years=10, ytm=0.035)
    print(f"  Price of a 10y, 3% coupon, $1000 bond at 3.5% YTM: ${price:.2f}")
    print("  (Trades below par because coupon < market yield.)")
    implied = bond_ytm(price=950, face=1000, coupon_rate=0.03, years=10)
    print(f"  If that bond instead trades at $950, implied YTM: {implied:.3%}")

    print()
    print(line)
    print("4. BLENDED-FINANCE LEVERAGE")
    print(line)
    ratio = mobilization_ratio(public_input=20, total_capital=100)
    print("  $20M concessional input inside a $100M project")
    print(f"    Mobilization ratio: {ratio:.1f}x private capital per public $")

    print()
    print(line)
    print("5. IMF RATE OF CHARGE (contrast: monetary, quota-funded)")
    print(line)
    basic = imf_rate_of_charge(sdr_rate=0.03, margin=0.01)
    surcharged = imf_rate_of_charge(
        sdr_rate=0.03, margin=0.01, level_surcharge=0.02, time_surcharge=0.01
    )
    print(f"  SDR rate 3% + 100 bps margin:        {basic:.2%}  (basic)")
    print(f"  + level + time surcharges:           {surcharged:.2%}  (large/long use)")
    print()
    print("  Note: the IMF funds this from POOLED MEMBER QUOTAS + SDRs,")
    print("  not from issuing bonds the way an MDB does. Different machine.")
    print(line)

--------------------------------------------------------------------
1. FUNDING-TO-LENDING SPREAD (the intermediation engine)
--------------------------------------------------------------------
  Bond yield (cost of funds): 3.50%
  Lending rate to borrower:   5.20%
  Gross spread:               170 bps

--------------------------------------------------------------------
2. GRANT ELEMENT (concessionality)
--------------------------------------------------------------------
  Bullet loan: $100M, 2% rate, 10y, repaid at maturity, disc 6%
    Grant element: 29.4%   (~29 cents/$ is a gift)

  IDA-style: $100M, 1% rate, 10y grace, 40y maturity, disc 5%
    Grant element: 54.8%

  Near-market: $100M, 5% rate, no grace, 15y, disc 5%
    Grant element: -0.0%   (~0 -> not concessional)

--------------------------------------------------------------------
3. BOND PRICING & YIELD (how the MDB raises money)
--------------------------------------------------------------------
  Price of a 10y, 3% 

# MDB Example

In [2]:
def npv_flat(cf, n, i):
    return sum(cf / (1 + i) ** n for n in range(1, n + 1))

In [3]:
interest = npv_flat(2, 10, 0.06)
print(f"The interest to be paid is ${interest:.2f}M.")

The interest to be paid is $14.72M.


In [4]:
loan_amt = 100
i = 0.06
n = 10
principal = loan_amt / (1 + i) ** n
print(f"The present value of the loan amount is ${principal:.2f}M.")

The present value of the loan amount is $55.84M.


In [5]:
debt_service = interest + principal
print(f"The total debt service is ${debt_service:.2f}M.")

The total debt service is $70.56M.


In [6]:
grant_element = (1 - debt_service / loan_amt) * 100
print(f"The grant element of this loan is {grant_element:.2f}%.")

The grant element of this loan is 29.44%.


In [23]:
# grant element formula
def grant_element(i, n, disc_rate, principal):
    annual_interest = principal * i
    interest_npv = sum(annual_interest / (1 + disc_rate) ** n for n in range(1, n + 1))
    principal_npv = principal / (1 + disc_rate) ** n
    debt_service = interest_npv + principal_npv
    return (1 - (debt_service / principal)) * 100

print(round(grant_element(0.02, 10, 0.06, 100),2))

29.44


In [24]:
print(round(grant_element(0.02, 40, 0.06, 100),2))

60.19


In [52]:
# to show the leverage of a mdb dollar where concessional financing mobilizes private capital by, for example, being in a first lost position
def mobilization_ratio(mdb_inv, priv_inv):
    return round((priv_inv / mdb_inv) * 100, 2)

print(mobilization_ratio(100, 120))

120.0


### Bonds to fund projects

# IMF Example

The IMF doesn't say "loan," "disburse," or "repay." A country purchases hard currency from the IMF using its own currency, and later repurchases its own currency back. So a "drawing" is a purchase and a "repayment" is a repurchase. It's an accounting fiction (the country is technically swapping assets), but it's why the documents read strangely. Interest is called the rate of charge.

Access is measured as a percentage of the country's quota — its capital subscription, which also sets its voting power. Everything scales off quota.

## Scenario

Say a middle-income country has a quota of SDR 1,000 million and negotiates a 4-year Extended Fund Facility (EFF) — the facility used for programs tied to structural reform — for SDR 4,000 million, i.e. 400% of quota. Assume it draws the full amount.

## The rate of charge

### Basic Rate

Basic rate = SDR interest rate + margin. The SDR interest rate (SDRi) is a floating benchmark reset weekly as a weighted average of 3-month money-market rates in the five basket currencies (USD, EUR, CNY, JPY, GBP). I couldn't pin down this exact week's figure — it moves every Friday and the IMF's live page is the place to check — so I'll use an illustrative 3.50%. For context, the SDRi ran near its 0.05% floor through 2021 and rose above 4% in 2023 as central banks tightened. The margin over the SDR interest rate is a verified-current 60 basis points — the Executive Board held it there for FY2025 and FY2026, the level set in October 2024.

So the basic rate = 3.50% + 0.60% = 4.10%, applied to the whole SDR 4,000M.

### Level-based surcharge

A level-based surcharge of 200 basis points applies to the portion of credit outstanding above 300% of quota — a threshold raised from 187.5% in the November 2024 reform. Here, 300% of quota = SDR 3,000M, so the surcharge hits only the top SDR 1,000M (the amount above threshold), not the whole loan. 

### Time-based surcharge

An additional 75-basis-point surcharge (reduced from 100 bps in the same reform) applies to that above-threshold portion once it has been outstanding beyond a set period — three years for most facilities, but 4¼ years under a longer-maturity Extended Arrangement. So this layer doesn't bite in the early years.

## Year-by-year charge

### Early years (< 4.25 years)

In [44]:
sdr = 1000
eff = 4000
sdr_rate = 0.035 #updated every Friday on imf.org
margin = 0.0060
basic_rate = sdr_rate + margin
level_surcharge_rate = 0.02

In [48]:
basic_charge = eff * basic_rate
print(f"The basic charge is SDR {basic_charge:.1f}M.")

The basic charge is SDR 164.0M.


In [49]:
level_surcharge = sdr * level_surcharge_rate
print(f"The level surcharge is SDR {level_surcharge:.1f}M.")

The level surcharge is SDR 20.0M.


In [50]:
early_total_charge = basic_charge + level_surcharge
print(f"The early phase total annual charge is SDR {early_total_charge:.1f}M.")

The early phase total annual charge is SDR 184.0M.


In [39]:
early_effective_rate = total_charge / eff
print(f"The early phase effective rate is {early_effective_rate:.2%}.")

The early phase effective rate is 4.60%.


### Later years (> 4.25 years)

In [51]:
time_surcharge_rate = 0.0075
time_surcharge = sdr * time_surcharge_rate
later_total_charge = early_total_charge + time_surcharge
later_effective_rate = later_total_charge / eff
print(f"The later phase total charge is SDR {later_total_charge:.1f}M.")
print(f"The later phase effective rate is {later_effective_rate:.2%}.")

The later phase total charge is SDR 191.5M.
The later phase effective rate is 4.79%.


#### Service and commitment fees

Two smaller items complete the picture. A service charge of 0.50% is levied on each amount drawn — one-time, so 0.50% × 4,000 = SDR 20M up front. And a refundable commitment fee is charged on amounts committed-but-not-yet-drawn; the reform raised the commitment-fee thresholds to 200% of quota annually and 600% cumulatively. If the country draws everything on schedule, the commitment fee is refunded and nets to roughly zero.

##### The setup
Same country: quota SDR 1,000M, an EFF of SDR 4,000M (400% of quota) over 4 years. EFF money isn't handed over all at once — it's phased, released in tranches tied to passing periodic program reviews. Say the schedule makes SDR 1,000M available each 12-month period (100% of quota per year).

The mechanic: at the start of each 12-month period, the IMF levies a commitment fee up front on the amount that could be drawn that period. As the country actually draws, the fee is refunded pro rata against what's drawn. Draw everything → full refund. Draw nothing → keep nothing back.

The fee is tiered by how much you're reserving relative to quota. The standard schedule is 15 bps on committed amounts up to the first threshold, 30 bps in the middle band, 60 bps at the top — and the November 2024 reform raised those thresholds to 200% of quota (annual) and 600% (cumulative). Since our country is reserving 100% of quota per year, it sits entirely in the bottom 15-bps tier.

Up-front fee each period = 0.15% × SDR 1,000M = SDR 1.5M (≈ US$2.0M at ~1.36 USD/SDR).

##### Branch A — the happy path (draws on schedule)
Reviews pass, tranches release, the country draws its full SDR 1,000M each period. Each year: SDR 1.5M charged at the start → fully refunded as the SDR 1,000M is drawn → net commitment cost ≈ zero. This is the "nets to roughly zero" case. The country pays interest (the rate of charge) on what it draws, but the reservation itself costs nothing because it used what it reserved.

##### Branch B — the program goes off track (the realistic case)
EFF programs frequently stall at a review — a fiscal target is missed, a reform slips, politics intervene. Play it out:
Period	Reserved	Fee charged up front	Drawn	Refund	Net cost
Year 1	1,000M	1.5M	1,000M	1.5M	0
Year 2	1,000M	1.5M	1,000M	1.5M	0
Year 3	1,000M	1.5M	0 (review fails)	0	1.5M
Year 4	—	—	—	—	—

In year 3 the country was charged SDR 1.5M for reserving the tranche, then couldn't complete the review, so nothing was drawn and nothing was refunded. That SDR 1.5M (~US$2M) is a pure dead-weight cost — it bought access the country never used. That's the fee doing its actual job: it prices the option of having Fund money on standby, and you forfeit the premium if you don't exercise it.

##### Branch C — deliberately precautionary
Now flip the intent. Some arrangements (a Stand-By Arrangement or a Flexible Credit Line) are taken as insurance — the country wants the backstop announced to markets but hopes never to draw. Here the "cost that nets to zero" becomes the whole point: if the country reserves SDR 1,000M/year for 4 years and draws nothing, it forfeits 4 × SDR 1.5M = SDR 6.0M (~US$8.2M) over the life of the arrangement. That's the explicit insurance premium for having a credible IMF line standing behind the currency. Cheap relative to a market crisis — which is exactly the sell.

The tiering bites when access is front-loaded
One wrinkle worth seeing: the tiers only matter if a single period's reservation is large. Suppose instead of level phasing, year 1 front-loaded SDR 3,000M of availability (300% of quota):
First 200% of quota (SDR 2,000M) @ 15 bps = SDR 3.0M
Next 100% (SDR 2,000M → 3,000M) @ 30 bps = SDR 3.0M
Up-front fee that period = SDR 6.0M (double the flat-phasing case)

So the reformed thresholds reward spreading access across periods and penalize cramming a big reservation into one window — another quiet nudge in the same direction as the surcharges: don't reserve or hold more than you need.

The clean takeaway: The commitment fee is a refundable reservation charge, so it's invisible when a program runs to plan and drawings match commitments — but it converts into a real cost precisely in the two cases that matter operationally: an off-track program (involuntary) or a precautionary arrangement (deliberate insurance).

### Repayment

Under an EFF, each drawing is repaid in equal semiannual installments beginning 4½ years and ending 10 years after that drawing. So the country has a long grace-like front end, then a stepped paydown — quite different from a bullet or a standard amortizing MDB loan.

### Converting SDR to dollars

Charges accrue in SDR. To express the burden in USD you apply the daily SDR/USD rate (recently in the neighborhood of 1 SDR ≈ 1.36 USD — again, look up the live figure). At that illustrative rate, the ~SDR 184M first-year charge is roughly US\\$250M, on a loan of about US\\$5.4bn.

### MDB versus IMF

Everything above is the **GRA (General Resources Account)** — the IMF's non-concessional window, funded from the pooled quota resources of members, not from bonds. For the poorest countries the IMF lends instead through the PRGT (Poverty Reduction and Growth Trust), and there the story flips entirely: PRGT lending carries zero interest and no surcharges. The gap between the SDR rate the IMF pays its PRGT loan-providers and the zero rate borrowers pay is a subsidy the Fund has to raise donor money to cover — which is exactly the financing strain your global-health-financing reading keeps running into. That subsidy-cost problem is the IMF's version of the DAH squeeze. Center For Global Development

So the through-line versus MDBs: the IMF prices off a floating reserve-asset benchmark plus deliberately escalating penalty tiers (to police the size and duration of use), funds itself from a quota pool rather than capital markets, and reserves genuine concessionality for a separate, donor-subsidized low-income window.

In [60]:
face_value = 100
coupon_rate = 0.04
coupon = face_value * coupon_rate
price_to_yield = 0.0415
term = 5
coupon_pmt_year = 2

In [61]:
def bond_value(face_value, coupon_rate, price_to_yield, term, coupon_pmt_year):
    coupon = face_value * coupon_rate / 2
    eff_price_to_yield = price_to_yield / 2
    n = term * coupon_pmt_year
    interest_npv = sum((coupon / (1 + eff_price_to_yield) ** n for n in range(1,  n + 1)))
    principal_npv = face_value / (1 + eff_price_to_yield) ** n
    return round(interest_npv + principal_npv, 2)

print(bond_value(100, 0.04, 0.0415, 5, 2))

99.33
